## Paths and executables

In [1]:
conda activate gapseq
gapseq -v
conda deactivate

gapseq version: 1.4.0
Sequence DB md5sum: 954ab58 (2024-11-06, Bacteria)

    The Archaea Sequence Database has not yet been downloaded/updated (default version).
    Get the latest version by running: 'bash ./src/update_sequences.sh Archaea' in your gapseq installation directory.
                        


In [2]:
# Path to new strains' genomes and the panmodel
label="clostridiales"
genomeFolder="00-fastas/new"
pangenome="01-pangenome/$label.faa"
panmodelReactionTable="02-pan_model/$label-all-Reactions.tbl"

# Path to results folder
workFolder="03-strain_models"
modelsFolder=$workFolder/models
matchesFolder=$workFolder/matches
mkdir -p $workFolder $matchesFolder $modelsFolder

# root folder of this analysis
root=$(pwd)

## Parameters

In [3]:
n_cores=14
diamond_bits=150
gapseq_bits=150
biomass="pos"
medium="media/P2_medium_an.csv"

## Derive strain GSMM

### Anticipatory character cleanup

In [4]:
find $genomeFolder/*.faa | xargs -I % bash -c 'cat % | tr -d "’" > tmp && mv tmp %'

### Identify homologous pangenome sequences

In [5]:
diamond makedb --in $pangenome -d $label

diamond v2.0.14.152 (C) Max Planck Society for the Advancement of Science
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 16
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Database input file: 01-pangenome/clostridiales.faa
Opening the database file...  [0.001s]
Loading sequences...  [0.138s]
Masking sequences...  [0.162s]
Writing sequences...  [0.113s]
Hashing sequences...  [0.008s]
Loading sequences...  [0s]
Writing trailer...  [0s]
Closing the input file...  [0s]
Closing the database file...  [0.01s]

Database sequences  114661
  Database letters  36283757
     Database hash  6fe471da9c4bafa8de2124d0885b2495
        Total time  0.434000s


In [6]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % \
diamond blastp -d $label -q $genomeFolder/%.faa -o $matchesFolder/%.tsv --min-score $diamond_bits

diamond v2.0.14.152 (C) Max Planck Society for the Advancement of Science
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 16
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Temporary directory: 03-strain_models/matches
#Target sequences to report alignments for: 25
Opening the database...  [0.048s]
Database: clostridiales.dmnd (type: Diamond database, sequences: 114661, letters: 36283757)
Block size = 2000000000
Opening the input file...  [0.001s]
Opening the output file...  [0s]
Loading query sequences...  [0.003s]
Masking queries...  [0.008s]
Algorithm: Double-indexed
Building query histograms...  [0.004s]
Allocating buffers...  [0s]
Loading reference sequences...  [0.039s]
Masking reference...  [0.164s]
Initializing temporary storage...  [0.001s]
Building reference histograms...  [0.105s]
Allocating buffers...  [0s]
Processing query bl

### Filter panmodel BLAST table

In [7]:
for g in $(dir -1 $genomeFolder | xargs basename -s .faa); do
    mkdir -p $modelsFolder/$g
    echo "Filtering reaction table of $g"
    python utils/filter_blast_table.py $panmodelReactionTable $matchesFolder/$g.tsv $modelsFolder/$g/filtered-Reactions.tbl
done

Filtering reaction table of C_acetobutylicum
Filtering reaction table of C_acetobutylicum_NCBI
Filtering reaction table of C_beijerinckii_13821
Filtering reaction table of C_beijerinckii_791


### Construct draft strain model

In [8]:
conda activate gapseq

In [9]:
gapseq_exc=$(which gapseq)

In [10]:
for g in $(dir -1 $genomeFolder | xargs basename -s .faa); do
    echo "Finding transporters for $g"
    gapseq find-transport -K $n_cores -b $gapseq_bits -f $modelsFolder/$g $genomeFolder/$g.faa \
    >> $modelsFolder/$g/log.log 2>> $modelsFolder/$g/log.err
done

Finding transporters for C_acetobutylicum
Finding transporters for C_acetobutylicum_NCBI
Finding transporters for C_beijerinckii_13821
Finding transporters for C_beijerinckii_791


In [11]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % -P $n_cores bash -c \
"""
echo 'Drafting a model for %'
$gapseq_exc draft -r $modelsFolder/%/filtered-Reactions.tbl \
-t $modelsFolder/%/%-Transporter.tbl \
-b $biomass -u $gapseq_bits -f $modelsFolder/% \
>> $modelsFolder/%/log.log 2>> $modelsFolder/%/log.err
"""

Drafting a model for C_acetobutylicum
Drafting a model for C_acetobutylicum_NCBI
Drafting a model for C_beijerinckii_13821
Drafting a model for C_beijerinckii_791


### Gapfill draft strain model

Remove those annoying `R_` prefixes popping up sometimes in reaction labels

In [12]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % -P $n_cores bash -c \
"""
echo 'Gapfilling the model of %'
$gapseq_exc fill -q -m $modelsFolder/%/filtered-Reactions.tbl-draft.RDS -n $medium \
-c $modelsFolder/%/filtered-Reactions.tbl-rxnWeights.RDS -g $modelsFolder/%/filtered-Reactions.tbl-rxnXgenes.RDS \
-f $modelsFolder/% >>$modelsFolder/%/log.log 2>>$modelsFolder/%/log.err
sed -i -e 's/R_EX_/EX_/g' $modelsFolder/%/filtered-Reactions.tbl.xml
"""

Gapfilling the model of C_acetobutylicum
Gapfilling the model of C_acetobutylicum_NCBI
Gapfilling the model of C_beijerinckii_13821
Gapfilling the model of C_beijerinckii_791


In [13]:
conda deactivate